# Editing models: Remove and Add/Replace (MODEL_LIST.md lists 2 and 3)

Every model of the two lists on a small fixed set of dev samples of the final dataset (`cryandrrich/editjudge`):
1. **Samples**: 30 dev samples per task, chosen by their request categories (`python -m src.editors samples`).
2. **Plan**: Qwen2.5-VL-7B reads each request: what to find, with its own box (remove, replace), what to keep
   (background, add), the existing thing a new one goes on (add: its anchor), and what to paint.
3. **Masks**: Grounding DINO boxes inside Qwen's box (Qwen's box when DINO finds none), SAM 2.1 large segments
   them (add: around the anchor, minus what to keep; background: inverted). Outputs keep only the masked region.
4. **Run** every model, each in its own process: remove = `lama`, `migan`, `rorem`, `objectclear`, `osor`;
   add_replace = `sdxl-inpaint` (mask) and `paint-by-inpaint`, `erasedraw` (instruction only, add samples).
   Outputs, seconds per image and peak GPU memory go to `out/editors/<model>/`. `MODEL_LIMIT` gives a model its
   own smoke run (5 samples) inside a task's full run, for models added later.
5. **Score** with the scorer (`mlp-siglip`) next to the human edits, and **report**: `out/editors/<task>/report.md`
   and contact sheets (team only: they show real people).

`SMOKE` is set per task: True is the smoke run of MODEL_LIST.md, 5 samples (does each model load in fp16, fit a
T4, give no black images, how long does it take, are the masks right); False is all 30 samples, after the
coordinator approves. Plan prompts per task (src/editors/plan.py:TASK_VERSIONS): remove plan_v2, add_replace
plan_v3.

How to run:
1. Inputs: the dataset `cryandrrich/editjudge`; for the scores, also the output of `scorer_train` (it holds
   `out/scorer/runs/mlp-siglip/head.pt`). Settings: GPU T4 x2, internet on. Push the code first.
2. Optional secret `HF_TOKEN` (Add-ons > Secrets): faster downloads.
3. **Save Version > Save & Run All (Commit)**. To go on after a stop, add this notebook's previous output as input:
   finished steps are skipped; plans, masks and outputs of another prompt version are made again. Add the output
   of the second smoke run (plan_v2) to reuse its remove results; the first one (plan_v1) is of no use.

Time (estimate before the first run): about 45 GB of downloads (Qwen2.5-VL-7B 16.6, RORem 6.9, ObjectClear 7.8,
SDXL Inpainting 7, SigLIP 3.5, DINO + SAM 1.8; each is deleted from the cache after its step), then about 40 min of
GPU work for the full run.

In [ ]:
import glob, json, os, shutil, subprocess, sys

REPO = "https://github.com/CryAndRRich/edit-judge.git"
BRANCH = "editors-remove-add/kien"
CODE = "/kaggle/working/edit-judge"
OUT = "/kaggle/working/out"

if os.path.exists(CODE):
    shutil.rmtree(CODE)
subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO, CODE], check=True)
print(subprocess.run(["git", "-C", CODE, "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

def find_dataset():
    """The input folder that holds pairs.jsonl and images/ (the final dataset)."""
    for pattern in ("/kaggle/input/*/pairs.jsonl", "/kaggle/input/*/*/pairs.jsonl", "/kaggle/input/*/*/*/pairs.jsonl"):
        found = sorted(glob.glob(pattern))
        if found:
            return os.path.dirname(found[0])
    raise SystemExit("no input with pairs.jsonl: add the dataset cryandrrich/editjudge")

def find_input_folders(tail, max_depth=6):
    """Folders under /kaggle/input whose path ends with `tail` (never inside a dataset's images/ folder)."""
    found = []
    for root, dirs, _ in os.walk("/kaggle/input"):
        if root.endswith("/" + tail):
            found.append(root)
            dirs[:] = []
        elif root.count("/") - 2 >= max_depth:
            dirs[:] = []
        else:
            dirs[:] = [name for name in dirs if name != "images"]
    return found

DATA = find_dataset()
os.environ["EJ_DATA_DIR"] = DATA            # read-only input
os.environ["EJ_OUTPUT_DIR"] = OUT           # all results are written here
os.environ["EJ_ENVS_DIR"] = "/tmp/ej-envs"
os.makedirs(f"{OUT}/editors", exist_ok=True)
os.chdir(CODE)

# an earlier output of this notebook (to go on where it stopped), and the scorer's trained heads
for folder in find_input_folders("out/editors"):
    shutil.copytree(folder, f"{OUT}/editors", dirs_exist_ok=True)
    print("earlier editors output copied from", folder)
runs = [f for f in find_input_folders("scorer/runs") if glob.glob(f"{f}/mlp-siglip/head.pt")]
if runs:
    shutil.copytree(runs[0], f"{OUT}/scorer/runs", dirs_exist_ok=True)
    print("scorer heads copied from", runs[0])
else:
    print("no scorer_train output among the inputs: the scoring step is skipped")

# Model weights go to /tmp: /kaggle/working is limited to 20 GB and is saved as the notebook's output
os.environ["HF_HOME"] = "/tmp/hf"
os.environ["EJ_MODELS_DIR"] = "/tmp/models"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    os.environ["HF_HUB_VERBOSITY"] = "error"

In [ ]:
!pip -q install -U "transformers>=4.56,<5" "diffusers>=0.35,<0.40" "peft>=0.14" accelerate sentencepiece opencv-python-headless
!nvidia-smi --query-gpu=name,memory.total --format=csv

def run_editors(*args, check=True):
    """python -m src.editors <args>: prints every line as it comes."""
    process = subprocess.Popen([sys.executable, "-u", "-m", "src.editors", *map(str, args)],
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="", flush=True)
    if process.wait() and check:
        raise subprocess.CalledProcessError(process.returncode, args)
    return process.returncode

def drop_cache(*repos):
    """Delete downloaded weights that are no longer needed (the disk holds about 70 GB)."""
    for repo in repos:
        shutil.rmtree(f"/tmp/hf/hub/models--{repo.replace('/', '--')}", ignore_errors=True)
    print(f"{shutil.disk_usage('/tmp').free / 1e9:.0f} GB free")

run_editors("list")

In [ ]:
# per task: True = smoke run (5 samples, MODEL_LIST.md), False = all 30 (after approval)
SMOKE = {"remove": False, "add_replace": False}
TASKS = ["remove", "add_replace"]
LIMIT = {task: (["--limit", 5] if SMOKE[task] else []) for task in TASKS}
MODELS = {"remove": ["lama", "migan", "rorem", "objectclear", "osor"],
          "add_replace": ["sdxl-inpaint", "paint-by-inpaint", "erasedraw"]}
# models added after a task's smoke run get their own smoke run first (5 samples), whatever SMOKE says, e.g.
# {"new-model": 5}; every model has passed its smoke run (2026-11-21)
MODEL_LIMIT = {}
WEIGHTS = {"rorem": ["LetsThink/RORem"], "objectclear": ["jixin0101/ObjectClear"],
           "osor": ["QinmingZhou/OSOR"],         # its SDXL-Inpainting base is shared with sdxl-inpaint: kept
           "sdxl-inpaint": ["diffusers/stable-diffusion-xl-1.0-inpainting-0.1"],
           "paint-by-inpaint": ["paint-by-inpaint/add-finetuned-mb"], "erasedraw": ["alpercanberk/erasedraw-v0.2"]}

In [ ]:
# 1-3. samples (fixed by the dataset: the same every time), plans (Qwen over both GPUs), masks
for task in TASKS:
    run_editors("samples", "--task", task)
for task in TASKS:
    run_editors("plan", "--task", task, *LIMIT[task])
drop_cache("Qwen/Qwen2.5-VL-7B-Instruct")
for task in TASKS:
    run_editors("masks", "--task", task, *LIMIT[task])
drop_cache("IDEA-Research/grounding-dino-base", "facebook/sam2.1-hiera-large")

In [ ]:
# 4. every model, each in its own process; a model that fails does not stop the others
failed = []
for task in TASKS:
    for model in MODELS[task]:
        limit = ["--limit", MODEL_LIMIT[model]] if model in MODEL_LIMIT else LIMIT[task]
        if run_editors("runall", "--task", task, "--models", model, *limit, check=False):
            failed.append(model)
        drop_cache(*WEIGHTS.get(model, []))
print("failed:", failed or "none")

In [ ]:
# 5. scores and reports
from IPython.display import Image, Markdown, display
for task in TASKS:
    if glob.glob(f"{OUT}/scorer/runs/mlp-siglip/head.pt"):
        run_editors("score", "--task", task, check=False)
    run_editors("report", "--task", task)
    display(Markdown(open(f"{OUT}/editors/{task}/report.md").read()))
    sheets = sorted(glob.glob(f"{OUT}/editors/{task}/sheets/*.jpg"))
    if sheets:
        display(Image(sheets[0]))

In [ ]:
# Last cell: the clone of the code is removed (the saved output keeps only results), and the whole out/ folder
# goes into one zip next to it: download /kaggle/working/out.zip. The output shows real people: team only.
os.chdir("/kaggle/working")
shutil.rmtree(CODE, ignore_errors=True)
archive = shutil.make_archive("/kaggle/working/out", "zip", "/kaggle/working", "out")
print(archive, f"{os.path.getsize(archive) / 1e6:.1f} MB")